# LTE-LoRA Alignment Training on Colab A100

**Project:** Reproducibility of "Learning to Edit: Aligning LLMs with Knowledge Editing" (ACL 2024)

This notebook trains LTE-LoRA on LLaMA2-Chat-7B using the authors' training data.

**Before running:** Make sure you have selected an A100 GPU runtime:
- Runtime > Change runtime type > A100 GPU

**Estimated time:** 4-8 hours on A100

## Step 1: Install Dependencies

Run these cells in order. flash_attn compiles from source (~10-15 min) but only once.

In [ ]:
!nvidia-smi
!echo "---"
!python --version

In [ ]:
# Step 1a: Core dependencies (fast — ~1 min)
!pip install transformers==4.44.0 accelerate peft bitsandbytes scipy deepspeed
!pip install fschat[model_worker,webui] sentence_transformers
print("\n✅ Core dependencies installed!")
print("Transformers version:", __import__('transformers').__version__)

In [ ]:
# Step 1b: flash_attn — install prebuilt for PyTorch 2.10 + CUDA 12.8
# Source compile fails on this combo, so we use the nightly/prebuilt approach
!pip install packaging ninja
!pip install flash-attn==2.7.4.post1 --no-build-isolation 2>/dev/null || \
    pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.5-cp312-cp312-linux_x86_64.whl 2>/dev/null || \
    echo "⚠️ flash_attn prebuilt not available — training will use PyTorch native SDPA (same performance on A100)"

# Check if it worked
try:
    import flash_attn
    print(f"✅ flash_attn {flash_attn.__version__} installed!")
except ImportError:
    print("⚠️ flash_attn not available — PyTorch SDPA will be used instead (this is fine)")

## Step 2: Clone LTE Repo & Download Data

In [ ]:
import os
os.chdir('/content')

!git clone https://github.com/YJiangcm/LTE.git
os.chdir('/content/LTE')

# Patch train_lora.py — fix compatibility with newer transformers + optional flash_attn
with open('FastChat/fastchat/train/train_lora.py', 'r') as f:
    code = f.read()

# Fix 1: "from transformers import deepspeed" was removed in transformers v5
code = code.replace(
    'from transformers import Trainer, BitsAndBytesConfig, deepspeed',
    '''from transformers import Trainer, BitsAndBytesConfig
try:
    from transformers import deepspeed
except ImportError:
    from transformers.integrations import deepspeed'''
)

# Fix 2: Make flash_attn import optional (fallback if not installed)
code = code.replace(
    '''from fastchat.train.llama_flash_attn_monkey_patch import (
    replace_llama_attn_with_flash_attn,
)''',
    '''try:
    from fastchat.train.llama_flash_attn_monkey_patch import (
        replace_llama_attn_with_flash_attn,
    )
except ImportError:
    print("flash_attn not installed - skipping")
    def replace_llama_attn_with_flash_attn():
        raise ImportError("flash_attn not installed")'''
)

# Fix 3: Make CPUAdam builder optional
code = code.replace(
    '''import deepspeed
deepspeed.ops.op_builder.CPUAdamBuilder().load()''',
    '''try:
    import deepspeed
    deepspeed.ops.op_builder.CPUAdamBuilder().load()
except Exception:
    print("CPUAdam builder not available - using default optimizer")'''
)

with open('FastChat/fastchat/train/train_lora.py', 'w') as f:
    f.write(code)

print("✅ Repo cloned and patched (3 fixes applied)!")

In [ ]:
# Download LTE training data from HuggingFace
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id='YuxinJiang/LTE_train_data',
    filename='LTE_train_data.json',
    repo_type='dataset',
    local_dir='data/'
)
print(f'Downloaded to: {path}')
print(f'File size: {os.path.getsize(path) / 1024 / 1024:.1f} MB')

# Verify
import json
with open('data/LTE_train_data.json') as f:
    data = json.load(f)
print(f'Training examples: {len(data)}')

## Step 3: Model Setup

We use the ungated mirror of LLaMA-2-Chat-7B, so **no HuggingFace token needed**.

In [ ]:
# No token needed — using ungated mirror
MODEL_NAME = "NousResearch/Llama-2-7b-chat-hf"
print(f"Will use model: {MODEL_NAME}")

In [ ]:
# Pre-download the model to avoid timeout during training
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "NousResearch/Llama-2-7b-chat-hf"

print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Downloading model (this takes a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
print(f"Model downloaded. Parameters: {model.num_parameters() / 1e9:.1f}B")
del model  # free memory before training
del tokenizer
torch.cuda.empty_cache()
print("Memory cleared. Ready for training.")

## Step 4: Run LTE-LoRA Training

Using the config from `FastChat/lora_train.sh` with reduced batch size to fit A100 40GB:
- LoRA r=8, alpha=16, dropout=0.05
- 3 epochs, lr=3e-4, cosine scheduler
- **batch_size=2, gradient_accumulation=64** (same effective batch = 128)

**This cell takes 4-8 hours. Don't close the tab!**

In [ ]:
import os
os.chdir('/content/LTE')
os.environ["WANDB_DISABLED"] = "true"

# Clear any leftover GPU memory from the model pre-download
import torch
torch.cuda.empty_cache()

# Run the LoRA training script
# batch_size reduced from 8→2 to fit A100 40GB, grad_accum increased 16→64 to keep effective batch=128
!python FastChat/fastchat/train/train_lora.py \
    --model_name_or_path NousResearch/Llama-2-7b-chat-hf \
    --lora_r 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --data_path data/LTE_train_data.json \
    --bf16 True \
    --output_dir output_lte_lora_llama-2_7b_chat \
    --num_train_epochs 3 \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 64 \
    --evaluation_strategy no \
    --save_strategy steps \
    --save_steps 300 \
    --save_total_limit 2 \
    --learning_rate 3e-4 \
    --weight_decay 0.0 \
    --warmup_ratio 0.03 \
    --lr_scheduler_type cosine \
    --logging_steps 1 \
    --tf32 True \
    --model_max_length 2048 \
    --gradient_checkpointing True \
    --q_lora False \
    --report_to none

## Step 4b: FALLBACK — QLoRA Training (if OOM on A100)

If the above cell OOMs, uncomment and run this cell instead.
This uses 4-bit quantization (QLoRA) which uses ~6-7 GB VRAM.

In [ ]:
# # FALLBACK: QLoRA (4-bit) — only run if Step 4 OOMs
# import os
# os.chdir('/content/LTE')
#
# !python FastChat/fastchat/train/train_lora.py \
#     --model_name_or_path NousResearch/Llama-2-7b-chat-hf \
#     --lora_r 8 \
#     --lora_alpha 16 \
#     --lora_dropout 0.05 \
#     --data_path data/LTE_train_data.json \
#     --bf16 True \
#     --output_dir output_lte_lora_llama-2_7b_chat \
#     --num_train_epochs 3 \
#     --per_device_train_batch_size 4 \
#     --per_device_eval_batch_size 4 \
#     --gradient_accumulation_steps 32 \
#     --evaluation_strategy no \
#     --save_strategy steps \
#     --save_steps 300 \
#     --save_total_limit 2 \
#     --learning_rate 3e-4 \
#     --weight_decay 0.0 \
#     --warmup_ratio 0.03 \
#     --lr_scheduler_type cosine \
#     --logging_steps 1 \
#     --tf32 True \
#     --model_max_length 2048 \
#     --gradient_checkpointing True \
#     --q_lora True

## Step 5: Verify Training Output & Save Checkpoint

In [ ]:
import os
output_dir = '/content/LTE/output_lte_lora_llama-2_7b_chat'

# List output files
print("Training output files:")
for f in sorted(os.listdir(output_dir)):
    full = os.path.join(output_dir, f)
    if os.path.isfile(full):
        size = os.path.getsize(full) / 1024 / 1024
        print(f"  {f} ({size:.1f} MB)")
    else:
        print(f"  {f}/")

In [ ]:
# Quick sanity check: load the trained LoRA adapter
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

MODEL_NAME = "NousResearch/Llama-2-7b-chat-hf"
LORA_PATH = "/content/LTE/output_lte_lora_llama-2_7b_chat"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, LORA_PATH)

# Test with an LTE-style prompt
prompt = """Please acknowledge the updated information provided below and respond to the subsequent query.

[Updated Information]:
1. The president of France is John Smith

[Query]:
Who is the president of France?"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)
print("Model response:", tokenizer.decode(outputs[0], skip_special_tokens=True))

## Step 6: Save to Google Drive

Save the checkpoint so it persists after the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy LoRA checkpoint to Google Drive
!mkdir -p /content/drive/MyDrive/LTE_checkpoints
!cp -r /content/LTE/output_lte_lora_llama-2_7b_chat /content/drive/MyDrive/LTE_checkpoints/
print("Checkpoint saved to Google Drive!")

# Verify
!ls -la /content/drive/MyDrive/LTE_checkpoints/output_lte_lora_llama-2_7b_chat/

---
## Day 3: Evaluation (run after training completes)

After training, download the sentence transformer model needed for retrieval, then run the evaluation scripts.

In [ ]:
# Download the retrieval model needed for IKE/LTE inference
os.chdir('/content/LTE')

!git clone https://huggingface.co/sentence-transformers/multi-qa-mpnet-base-dot-v1 SeqEdit/multi-qa-mpnet-base-dot-v1
!git clone https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2 hugging_cache/all-MiniLM-L6-v2

print("Retrieval models downloaded.")

### Update hparams to point to the trained model

In [ ]:
# Update the IKE hparams to point to our trained LTE model
import yaml

hparams_path = '/content/LTE/EasyEdit/hparams/IKE/llama-7b.yaml'
with open(hparams_path, 'r') as f:
    hparams = yaml.safe_load(f)

print("Original hparams:")
print(hparams)

# Update model path to point to our trained LTE-LoRA model
hparams['model_name'] = 'NousResearch/Llama-2-7b-chat-hf'
hparams['lora_name'] = '/content/LTE/output_lte_lora_llama-2_7b_chat'
hparams['sentence_model_name'] = '/content/LTE/hugging_cache/all-MiniLM-L6-v2'

with open(hparams_path, 'w') as f:
    yaml.dump(hparams, f, default_flow_style=False)

print("\nUpdated hparams:")
print(hparams)

### Run LTE-LoRA Evaluation on ZsRE

In [ ]:
os.chdir('/content/LTE/EasyEdit')

# Evaluate on ZsRE — all aspects
for aspect in ['portability_r', 'portability_s', 'portability_l', 'locality_rs']:
    print(f"\n{'='*60}")
    print(f"Evaluating ZsRE — {aspect}")
    print(f"{'='*60}")
    !python run_knowedit.py \
        --editing_method=IKE \
        --hparams_dir=hparams/IKE/llama-7b \
        --data_dir=data/knowedit/ZsRE/ZsRE-test-all.json \
        --eval_aspect=$aspect \
        --fluency

### Run LTE-LoRA Evaluation on WikiDatacounterfact

In [ ]:
os.chdir('/content/LTE/EasyEdit')

# Evaluate on WikiDatacounterfact — all aspects
for aspect in ['portability_r', 'portability_s', 'portability_l', 'locality_rs', 'locality_f']:
    print(f"\n{'='*60}")
    print(f"Evaluating WikiData_cf — {aspect}")
    print(f"{'='*60}")
    !python run_knowedit.py \
        --editing_method=IKE \
        --hparams_dir=hparams/IKE/llama-7b \
        --data_dir=data/knowedit/wiki_counterfact/test_cf.json \
        --eval_aspect=$aspect \
        --fluency

### Collect All Metrics

In [ ]:
import json
import os

output_dir = '/content/LTE/EasyEdit/output'

print("\n" + "="*80)
print("LTE-LoRA RESULTS SUMMARY")
print("="*80)

for dataset_label, data_key in [('ZsRE', 'ZsRE'), ('WikiData_cf', 'wiki_counterfact')]:
    print(f"\n--- {dataset_label} ---")
    
    for fname in sorted(os.listdir(output_dir)):
        if data_key in fname and fname.endswith('.json'):
            with open(os.path.join(output_dir, fname)) as f:
                metrics = json.load(f)
            
            # Extract aspect from filename
            aspect = fname.replace(f'IKE_{data_key}_', '').replace('_results.json', '')
            
            # Calculate metrics
            edit_success = sum(m['post']['rewrite_acc'] for m in metrics) / len(metrics) * 100
            
            print(f"  {aspect}:")
            print(f"    Edit Success: {edit_success:.2f}%")
            print(f"    N examples: {len(metrics)}")
            
            # Print portability/locality
            if 'portability' in metrics[0]['post'] and metrics[0]['post']['portability']:
                for key, val in metrics[0]['post']['portability'].items():
                    avg = sum(m['post']['portability'][key] for m in metrics) / len(metrics) * 100
                    print(f"    Portability ({key}): {avg:.2f}%")
            if 'locality' in metrics[0]['post'] and metrics[0]['post']['locality']:
                for key, val in metrics[0]['post']['locality'].items():
                    avg = sum(m['post']['locality'][key] for m in metrics) / len(metrics) * 100
                    print(f"    Locality ({key}): {avg:.2f}%")
            if 'fluency' in metrics[0]['post']:
                fluency = sum(m['post']['fluency']['ngram_entropy'] for m in metrics) / len(metrics) * 100
                print(f"    Fluency: {fluency:.2f}")

print("\n" + "="*80)
print("Copy these numbers into the shared spreadsheet!")
print("="*80)

### Save Results to Google Drive

In [ ]:
# Save all results to Drive
!mkdir -p /content/drive/MyDrive/LTE_checkpoints/results
!cp -r /content/LTE/EasyEdit/output/* /content/drive/MyDrive/LTE_checkpoints/results/
print("Results saved to Google Drive!")